# Tratando os dados de internações

Este notebook consolida as **internações por doenças respiratórias** (SUS) no **município do Rio de Janeiro** e prepara a **série diária de internações** para modelagem.
O foco é: leitura dos dados brutos anuais, filtros clínicos e geográficos coerentes com exposição ambiental e uma **análise exploratória inicial**.

> **Escopo**: 2012–2025, diagnósticos respiratórios (CID-10 selecionados), residentes da cidade do Rio e permanência hospitalar > 0.

## Objetivos desta etapa
1. **Carregar** todos os parquet anuais (2012–2025) diretamente do GitHub (*raw*).
2. **Padronizar** campos-chave (datas, CID-10) e **filtrar** registros de interesse.
3. **Construir** a série **diária** de internações (`data_dia` × `internacoes`).

## Configurações e importações
Bibliotecas usadas e parâmetros gerais.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import pandas as pd

## Carregamento e Unificação das Estações

In [2]:
anos = range(2012, 2013)
tpl = "https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/refs/heads/Refactoring-And-Documentation/Data/RawData/DataSus/dados_filtrados_{ano}.parquet"

dfs = []
for ano in anos:
    url = tpl.format(ano=ano)
    print(f"Lendo: {url}")
    df = pd.read_parquet(url)
    df["ano_arquivo"] = ano
    dfs.append(df)

df_sus = pd.concat(dfs, ignore_index=True)

Lendo: https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/refs/heads/Refactoring-And-Documentation/Data/RawData/DataSus/dados_filtrados_2012.parquet


## Inspeção inicial

Visão geral de dimensões, tipos, amostra e dados ausentes. A coluna de data principal
virá de `DT_INTER` (quando numérica tipo `YYYYMMDD`) ou de `data_formatada` (se já estiver disponível).

In [4]:
print("Dimensões:", df_sus.shape)

print("\nTipos:")
print(df_sus.dtypes)

print("\nAmostra:")
display(df_sus.head(5))

print("\nValores ausentes por coluna (top 20):")
na_counts = df_sus.isna().sum().sort_values(ascending=False)
na_perc   = (df_sus.isna().mean()*100).round(2).sort_values(ascending=False)
display(pd.DataFrame({"NA_count": na_counts, "NA_%": na_perc}).head(20))

Dimensões: (59708, 16)

Tipos:
MUNIC_RES         string[python]
NASC              string[python]
SEXO                       Int64
DT_INTER          string[python]
DT_SAIDA          string[python]
DIAG_PRINC        string[python]
DIAG_SECUN        string[python]
IDADE             string[python]
COD_IDADE         string[python]
RACA_COR          string[python]
ETNIA             string[python]
DIAS_PERM         string[python]
MORTE             string[python]
CID_MORTE         string[python]
data_formatada            object
ano_arquivo                int64
dtype: object

Amostra:


,MUNIC_RES,NASC,SEXO,DT_INTER,DT_SAIDA,DIAG_PRINC,DIAG_SECUN,IDADE,COD_IDADE,RACA_COR,ETNIA,DIAS_PERM,MORTE,CID_MORTE,data_formatada,ano_arquivo
0,330455,19311020,1,20120101,20120210,J189,,80,4,03,0000,40,1,J960,2012-01-01,2012
1,330250,20101005,1,20120101,20120103,J40,,1,4,99,0000,2,0,,2012-01-01,2012
2,330210,19540705,3,20120101,20120103,J189,,57,4,01,0000,2,0,,2012-01-01,2012
3,330455,19330422,3,20120101,20120131,J849,,78,4,03,0000,30,0,,2012-01-01,2012
4,521150,19630706,3,20120101,20120105,J440,,48,4,99,0000,4,0,,2012-01-01,2012



Valores ausentes por coluna (top 20):


,NA_count,NA_%
MUNIC_RES,0,0.0
NASC,0,0.0
SEXO,0,0.0
DT_INTER,0,0.0
DT_SAIDA,0,0.0
DIAG_PRINC,0,0.0
DIAG_SECUN,0,0.0
IDADE,0,0.0
COD_IDADE,0,0.0
RACA_COR,0,0.0


## Filtrando dados de saúde

### Filtrando por CID-10 de Interesse

Nesta etapa, serão filtrados os códigos CID-10 relevantes para nossa análise:

**Códigos selecionados:**
- **Pneumonia:** J18, J15
- **Insuficiência Respiratória:** J21, J96  
- **Bronquites:** J40, J41, J42
- **Enfisema/DPOC:** J43, J44
- **Asma:** J45, J46

In [23]:
cids_associados_a_poluicao = {"J15","J18","J21","J96","J40","J41","J42","J43","J44","J45","J46"}

# Limpando a coluna DIAG_PRINC: convertendo para string, maiúsculas, removendo espaços e caracteres não alfanuméricos
df_sus["DIAG_PRINC"] = (df_sus["DIAG_PRINC"].astype(str).str.upper().str.strip().str.replace(r"[^A-Z0-9]", "",regex=True))

# Extraindo os primeiros 3 caracteres como prefixo CID
df_sus["CID_PREFIXO"] = df_sus["DIAG_PRINC"].str[:3]

# Filtrando o dataframe para incluir apenas linhas com prefixos CID relevantes
sus_filtrado = df_sus[df_sus["CID_PREFIXO"].isin(cids_associados_a_poluicao)].copy()

print(f"Filtered data shape: {sus_filtrado.shape}")
print("\nCID prefix counts:")
print(sus_filtrado["CID_PREFIXO"].value_counts())
print("\nFirst 5 rows of filtered data:")
display(sus_filtrado.head())

Filtered data shape: (44607, 17)

CID prefix counts:
CID_PREFIXO
J18    17471
J15    10694
J45     4590
J44     2967
J96     2719
J21     2717
J40     1671
J46      846
J43      698
J41      165
J42       69
Name: count, dtype: int64

First 5 rows of filtered data:


,MUNIC_RES,NASC,SEXO,DT_INTER,DT_SAIDA,DIAG_PRINC,DIAG_SECUN,IDADE,COD_IDADE,RACA_COR,ETNIA,DIAS_PERM,MORTE,CID_MORTE,data_formatada,ano_arquivo,CID_PREFIXO
0,330455,19311020,1,20120101,20120210,J189,,80,4,03,0000,40,1,J960,2012-01-01,2012,J18
1,330250,20101005,1,20120101,20120103,J40,,1,4,99,0000,2,0,,2012-01-01,2012,J40
2,330210,19540705,3,20120101,20120103,J189,,57,4,01,0000,2,0,,2012-01-01,2012,J18
4,521150,19630706,3,20120101,20120105,J440,,48,4,99,0000,4,0,,2012-01-01,2012,J44
6,330452,19470310,3,20120101,20120105,J189,,64,4,03,0000,4,0,,2012-01-01,2012,J18


### Filtrando por Municípios de Interesse

Nesta etapa, serão filtrados os municípios relevantes para nossa análise:

#### Municípios Selecionados

- **330455** - Rio de Janeiro
- **330510** - São João de Meriti
- **330320** - Nilópolis
- **330285** - Mesquita
- **330045** - Belford Roxo
- **330350** - Nova Iguaçu
- **330170** - Duque de Caxias

In [24]:
municipios_interesse = ['330455','330510','330320','330285','330045','330350','330170']

sus_filtrado = sus_filtrado[sus_filtrado['MUNIC_RES'].isin(municipios_interesse)]

print("Dimensões após filtro por municípios:", sus_filtrado.shape)

Dimensões após filtro por municípios: (15228, 17)


### Filtrando por dias de permanência maior que zero

In [28]:
sus_filtrado['DIAS_PERM'] = pd.to_numeric(sus_filtrado['DIAS_PERM'], errors='coerce')

sus_filtrado = sus_filtrado[sus_filtrado['DIAS_PERM'] > 0]

print("Dimensões após filtro por permanência:", sus_filtrado.shape)

Dimensões após filtro por permanência: (15063, 17)



## Série diária de internações


### Corrigindo data

In [7]:
sus_filtrado = sus_filtrado.copy()
sus_filtrado["data_dia"] = pd.to_datetime(sus_filtrado["data_formatada"], errors="coerce").dt.normalize()
sus_filtrado = sus_filtrado.dropna(subset=["data_dia"])

### Gerando quantidade de internações por dia

In [8]:
internacoes_diario = (
    sus_filtrado.groupby("data_dia")
    .size()
    .rename("num_internacoes")
    .astype("int64")
    .reset_index()
    .sort_values("data_dia")
)

### Preenchendo calendário
Preencher calendário (datas faltantes -> 0 internações)

In [9]:
idx_full = pd.date_range(internacoes_diario["data_dia"].min(),
                         internacoes_diario["data_dia"].max(),
                         freq="D")

internacoes_rj = (
    internacoes_diario.set_index("data_dia")
    .reindex(idx_full, fill_value=0)
    .rename_axis("data_dia")
    .reset_index()
)

### Gerando CSV de saída

In [10]:
project_root = Path().resolve().parents[2]  
output_dir = project_root / "data" / "datasus"
output_dir.mkdir(parents=True, exist_ok=True)

output_csv_path = output_dir / "INTERNACOES_DOENCA_RESP_RJ.csv"
internacoes_rj.to_csv(output_csv_path, index=False, encoding="utf-8")

# Contagem de dias com 0 internações
qtd_zeros = int((internacoes_rj["num_internacoes"] == 0).sum())
print(f"Arquivo salvo em: {output_csv_path}")
print(f"Dias com 0 internações: {qtd_zeros}")
print(internacoes_rj.head())

Arquivo salvo em: D:\João Henrique\OneDrive\Área de Trabalho\qualiar\data\datasus\INTERNACOES_DOENCA_RESP_RJ.csv
Dias com 0 internações: 0
    data_dia  num_internacoes
0 2012-01-01              507
1 2012-01-02              728
2 2012-01-03              429
3 2012-01-04              416
4 2012-01-05              403


In [11]:
display(internacoes_rj[["data_dia", "num_internacoes"]].head(10))

,data_dia,num_internacoes
0,2012-01-01,507
1,2012-01-02,728
2,2012-01-03,429
3,2012-01-04,416
4,2012-01-05,403
5,2012-01-06,520
6,2012-01-07,299
7,2012-01-08,390
8,2012-01-09,442
9,2012-01-10,546
